In [1]:
#Engineering % to seasons best as a output measurment

In [2]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

In [7]:
#Load columns needed

cols_needed = [
    'RaceId',
    'Date',
    'AthleteName',
    'AthleteGenderName',
    'RaceEventName',
    'RaceEventDistance',
    'RaceEventSwimmingStyleName',
    'RaceEventIsRelay',
    'PoolLength',
    'PhaseName',
    'RaceTime',
    'CompetitionName']

df = pd.read_csv('Nemo.csv', usecols = cols_needed)
print(df.shape)
df.head()

(2837, 12)


,RaceId,PoolLength,Date,RaceEventDistance,RaceEventIsRelay,RaceEventName,RaceEventSwimmingStyleName,AthleteName,AthleteGenderName,CompetitionName,PhaseName,RaceTime
0,003223f3-ae33-4dba-8fc9-365f7356ac4f,25,14/12/2024,50,False,50 m Breaststroke,Breaststroke,Angharad Evans,Women,World Championships,Heat,29.92
1,0066b686-e4f1-414e-b79d-31bbbf61ef55,50,19/11/2025,200,False,200 m Freestyle,Freestyle,Freya Anderson,Women,Training,Final,87.10
2,00721f02-0a18-4031-bdc8-bdad872a41b2,50,30/07/2022,200,False,200 m Freestyle,Freestyle,Evan Jones,Men,Commonwealth Games,Heat,109.00
3,009e1505-8709-4103-9e30-6be7c52e5f02,50,05/04/2024,200,False,200 m Backstroke,Backstroke,Holly McGill,Women,British Championships,Heat,132.12
4,00c56931-1e8d-438e-a2e1-86573d83878a,50,17/04/2025,100,False,100 m Backstroke,Backstroke,Dean Fearn,Men,British Championships,Heat,55.00


In [4]:
#INITIAL DATA CHECKS

In [8]:
#Check data types and missing values across columns
print(df.dtypes)
print("\nMissing values per column:")
print(df.isnull().sum())

RaceId                            str
PoolLength                      int64
Date                              str
RaceEventDistance               int64
RaceEventIsRelay                 bool
RaceEventName                     str
RaceEventSwimmingStyleName        str
AthleteName                       str
AthleteGenderName                 str
CompetitionName                   str
PhaseName                         str
RaceTime                      float64
dtype: object

Missing values per column:
RaceId                        0
PoolLength                    0
Date                          0
RaceEventDistance             0
RaceEventIsRelay              0
RaceEventName                 0
RaceEventSwimmingStyleName    0
AthleteName                   0
AthleteGenderName             0
CompetitionName               0
PhaseName                     0
RaceTime                      0
dtype: int64


In [10]:
#Check unique values in categorical columns
##Sanity checks for poollength, relay, phasename, gender, and comp columns
###Will need to filter out unnecessary "competition names"

for col in ['PoolLength', 'RaceEventIsRelay', 'PhaseName', 'AthleteGenderName', 'CompetitionName']:
    print(f"\n--- {col} ---")
    print(df[col].value_counts())


--- PoolLength ---
PoolLength
50    2379
25     458
Name: count, dtype: int64

--- RaceEventIsRelay ---
RaceEventIsRelay
False    2581
True      256
Name: count, dtype: int64

--- PhaseName ---
PhaseName
Heat           1262
Final          1185
B-Final         136
Semi-Final      131
Time Trial       96
C-Final          23
Round of 16       2
D-Final           2
Name: count, dtype: int64

--- AthleteGenderName ---
AthleteGenderName
Women    1603
Men      1234
Name: count, dtype: int64

--- CompetitionName ---
CompetitionName
British Championships                   386
Edinburgh International                 358
Scottish Short Course                   290
Scottish Nationals                      180
Time Trial                              173
BUCS Championships                      166
Sette Colli                             156
European Championships                  155
World Championships                     150
AP International                        148
British Summer Championships 

In [11]:
#Check racetime distribution
##Checks for missing values, implausible values, gen. distribution

print(df['RaceTime'].describe())
print(f"\nZero or negative RaceTimes: {(df['RaceTime'] <= 0).sum()}")
print(f"Missing RaceTimes: {df['RaceTime'].isnull().sum()}")

count    2837.000000
mean       99.086567
std        82.376016
min         0.000000
25%        52.700000
50%        65.910000
75%       127.450000
max      1032.110000
Name: RaceTime, dtype: float64

Zero or negative RaceTimes: 100
Missing RaceTimes: 0


In [13]:
#Check date format

##Convert Date to datetime if not already
df['Date'] = pd.to_datetime(df['Date'])

#Confirm conversion worked
print(df['Date'].dtype)
print(f"Date range: {df['Date'].min()} to {df['Date'].max()}")

datetime64[us]
Date range: 2022-01-20 00:00:00 to 2025-12-31 00:00:00


In [15]:
#Filter and clean
##Remove unnecessary rows
###Includes relay races, junk comp types, missing race_times, phase filtering

#Inspect competitions where race time is missing

missing_rt = df[df['RaceTime'].isnull()]
print(missing_rt['CompetitionName'].value_counts())

Series([], Name: count, dtype: int64)
